# Short Term Memory

In [4]:
import os
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai.chat_models import ChatOpenAI

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="deepseek/deepseek-v4-flash"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're an assistant who's good at {ability}. Respond in 20 words or fewer"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
base_chain = prompt | llm

history = []

data = base_chain.invoke({"ability": "math", "input": "What's a right-angled triangle?", "history": history})

print(data.content)


A triangle with one 90-degree right angle.


In [5]:
history = [
    ("human", "What's a right-angled triangle?"),
    ("ai", data.content)
]

# Now the model receives the previous exchange as part of the prompt
base_chain.invoke({"ability": "math", "input": "What are the other types?", "history": history})

AIMessage(content='Acute (all angles <90°), obtuse (one >90°), plus equilateral, isosceles, and scalene by sides.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 310, 'prompt_tokens': 75, 'total_tokens': 385, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 279, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.00019725}, 'model_provider': 'openai', 'model_name': 'deepseek/deepseek-v4-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'rqsty-cmpl-00f8cb0a-4606-4899-ad3c-69acc104d5ec', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0aa6a-27ce-76b2-88df-d23894220abd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 310, 'total_tokens': 385, 'input_toke

# Configurable

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import ConfigurableField

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model_name="deepseek/deepseek-v4-flash"
).configurable_fields(
    model_name=ConfigurableField(
        description="The model to use for the chat",
        id="model",
        name="model",
    )
)

prompt = PromptTemplate.from_template("Write a Haiku on {subject}")
chain = prompt | llm

# Force the chain to use GPT-4 for this invocation
result_gpt4 = chain.with_config(configurable={"model_name": "gpt-oss-120b"}).invoke({"subject": "cat"})
print(result_gpt4)


content='Soft paws on the floor  \nWhiskers twitch in morning sun  \nPurrs by the warm hearth' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 404, 'prompt_tokens': 36, 'total_tokens': 440, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 382, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.0002478}, 'model_provider': 'openai', 'model_name': 'deepseek/deepseek-v4-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'rqsty-cmpl-9f03e732-7a82-4f1c-8ea0-c9d97938b915', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0aa75-6cc5-7ae1-b806-03015c0f7f4e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 404, 'total_tokens': 440, 'input_token_details': {'cache_read': 

# Persistance memory with Redis

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai.chat_models import ChatOpenAI

from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt + model (same as before)
model = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're an assistant who's good at {ability}. Respond in 20 words or fewer"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
base_chain = prompt | model

# Redis connection for persisting message history
REDIS_URL = "redis://localhost:6379/0"

def get_message_history(session_id: str) -> RedisChatMessageHistory:
    return RedisChatMessageHistory(session_id, url=REDIS_URL)

# Wrap the base chain so it reads/writes history automatically
redis_chain = RunnableWithMessageHistory(
    base_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="history",
)

/tmp/ipykernel_6583/4273227118.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import RedisChatMessageHistory
/home/ajayp/anaconda3/envs/langchain/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
# Create (or reuse) a math session and ask about cosine
redis_chain.invoke(
    {"ability": "math", "input": "What does cosine mean?"},
    config={"configurable": {"session_id": "math-thread1"}},
)

# Follow up in the same session; the chain will include the stored history
redis_chain.invoke(
    {"ability": "math", "input": "Tell me more!"},
    config={"configurable": {"session_id": "math-thread1"}},
)

# Create a separate physics session
redis_chain.invoke(
    {"ability": "physics", "input": "What is the theory of relativity?"},
    config={"configurable": {"session_id": "phy-thread1"}},
)

# Follow up in the physics session
redis_chain.invoke(
    {"ability": "physics", "input": "Tell me more!"},
    config={"configurable": {"session_id": "phy-thread1"}},
)

AIMessage(content='Special relativity: time slows, lengths contract at high speeds. General relativity: mass curves spacetime, producing gravity.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 394, 'prompt_tokens': 140, 'total_tokens': 534, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 359, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 94, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.00024471}, 'model_provider': 'openai', 'model_name': 'gpt-oss-120b', 'system_fingerprint': None, 'id': 'rqsty-cmpl-c178cd65-d0d0-4d9e-8689-91b3ea404c59', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ab04-aabe-7ac2-9884-f2933472d98c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 140, 'output_tokens': 394, 'total_tokens': 534, 'input_token_details